# Debug Scraper Memox

Local HTML first, then optional live list/article checks. Output list CSV: `csv/memox_list_debug.csv`.

In [1]:
from pathlib import Path
import sys
import importlib
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scrapers').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scrapers.common import cutoff_date, parse_date, records_to_df
import scrapers.memox as scraper
scraper = importlib.reload(scraper)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 160)

SAMPLE_PATH = PROJECT_ROOT / 'html_samples/memox.html'
OUTPUT_PATH = PROJECT_ROOT / 'csv/memox_list_debug.csv'
MAX_PAGES = 200
TITLE_LIMIT = 90
cutoff = cutoff_date()

print('project:', PROJECT_ROOT)
print('module:', scraper.__file__)
print('sample:', SAMPLE_PATH)
print('cutoff:', cutoff.date())


project: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project
module: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/scrapers/memox.py
sample: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/html_samples/memox.html
cutoff: 2026-02-28


In [2]:

def short_title(value, limit=TITLE_LIMIT):
    text = '' if pd.isna(value) else str(value).strip()
    return text if len(text) <= limit else text[: limit - 3] + '...'


def ensure_debug_columns(df):
    df = df.copy()
    if 'page_num' not in df.columns:
        df['page_num'] = pd.NA
    if 'published_date' not in df.columns:
        df['published_date'] = None
    df['published_dt'] = df['published_date'].apply(parse_date)
    return df


def print_debug_rows(df):
    for _, row in df.iterrows():
        page = row.get('page_num')
        page_text = '---' if pd.isna(page) else f'{int(page):03d}'
        date_text = row.get('published_date')
        date_text = 'None' if pd.isna(date_text) else str(date_text)
        print(f"page={page_text} | date={date_text} | title={short_title(row.get('title'))}")


def audit_list(df):
    df = ensure_debug_columns(df)
    print('total rows:', len(df))
    if len(df) == 0:
        print('No rows found.')
        return df
    print('first page:', df['page_num'].dropna().min() if df['page_num'].notna().any() else None)
    print('last page:', df['page_num'].dropna().max() if df['page_num'].notna().any() else None)
    print('newest date:', df['published_dt'].max())
    print('oldest date:', df['published_dt'].min())
    print('cutoff date:', cutoff)
    print('reached cutoff:', bool((df['published_dt'].dropna() < cutoff).any()))
    print('null parsed dates:', int(df['published_dt'].isna().sum()))

    print('\nrows per month:')
    display(
        df.assign(month=df['published_dt'].dt.to_period('M').astype(str))
        .groupby('month', dropna=False)
        .size()
        .reset_index(name='count')
    )

    print('\nrows per page:')
    display(
        df.groupby('page_num', dropna=False)
        .agg(rows=('url', 'count'), newest=('published_dt', 'max'), oldest=('published_dt', 'min'))
        .reset_index()
        .tail(80)
    )
    return df


In [3]:

def parse_local_list(html_text):
    soup = BeautifulSoup(html_text, 'html.parser')
    rows = []
    for card in soup.select('article'):
        title_tag = card.select_one('h2.entry-title a[href], h3 a[href]')
        link_tag = title_tag or card.select_one('a.post-thumbnail[href]')
        date_tag = card.select_one('time.entry-date.published, time.entry-date')
        excerpt_tag = card.select_one('.entry-content, .entry-summary')
        image_tag = card.select_one('img')
        if not title_tag or not link_tag:
            continue
        rows.append({
            'page_num': 1,
            'list_page_url': scraper.BASE_URL,
            'title': title_tag.get_text(' ', strip=True),
            'url': urljoin(scraper.BASE_URL, link_tag['href']),
            'published_date': (date_tag.get('datetime') or date_tag.get_text(' ', strip=True)) if date_tag else None,
            'excerpt': excerpt_tag.get_text(' ', strip=True) if excerpt_tag else None,
            'image_url': (image_tag.get('data-src') or image_tag.get('src')) if image_tag else None,
        })
    return records_to_df(rows)


## Local HTML list parse

In [4]:
html_text = SAMPLE_PATH.read_text(errors='replace')
local_df = parse_local_list(html_text)
local_df = ensure_debug_columns(local_df)
print_debug_rows(local_df)
local_df = audit_list(local_df)


page=001 | date=2026-06-26T13:15:52+07:00 | title=Usai Diresmikan Kapolres Malang Ratusan Jamaah Sholat Jumat Datangi Masjid Budi Trianti...
page=001 | date=2026-06-25T15:51:59+07:00 | title=Sambut Hari Bhayangkara ke-80 Polres Malang Bangun 7 Sumur Bor di 3 Kecamatan
page=001 | date=2026-06-24T21:01:54+07:00 | title=Giat Donor Darah Hari Bhayangkara ke-80 di Polres Malang Diikuti TNI dan Masyarakat Umum
page=001 | date=2026-06-24T20:52:40+07:00 | title=Mengaku Ajudan Gubenur Jatim, 2 Tersangka Kasus Penipuan Program UMKM Diamankan Polres ...
page=001 | date=2026-06-23T12:06:27+07:00 | title=Jelang Hari Bhayangkara ke-80, Kapolres Malang Pimpin Giat Ziarah dan Tabur Bunga di TMP
page=001 | date=2026-06-20T21:56:16+07:00 | title=Polres Malang Jaring Atlet Esport Lewat Turnamen Mobile Legends Kapolres Malang Cup 2026
page=001 | date=2026-06-19T20:00:16+07:00 | title=Jajaran Polsek Polres Malang Ramaikan Kapolres Malang Cup Jelang Hari Bhayangkara ke-80
page=001 | date=2026-06-19T19:57:15

,month,count
0,2026-05,5
1,2026-06,31



rows per page:


,page_num,rows,newest,oldest
0,1,36,2026-06-26 13:15:52,2026-05-26 20:36:41


## Live list scrape

In [5]:
live_df = scraper.scrape_list(cutoff, max_pages=MAX_PAGES)
live_df = ensure_debug_columns(live_df)
print_debug_rows(live_df)
live_df = audit_list(live_df)


Scraping Memox page 1: https://memox.co.id/tag/kabupaten-malang/
Scraping Memox page 2: https://memox.co.id/tag/kabupaten-malang/page/2/
Scraping Memox page 3: https://memox.co.id/tag/kabupaten-malang/page/3/
Scraping Memox page 4: https://memox.co.id/tag/kabupaten-malang/page/4/
Scraping Memox page 5: https://memox.co.id/tag/kabupaten-malang/page/5/
Scraping Memox page 6: https://memox.co.id/tag/kabupaten-malang/page/6/
Scraping Memox page 7: https://memox.co.id/tag/kabupaten-malang/page/7/
Scraping Memox page 8: https://memox.co.id/tag/kabupaten-malang/page/8/
Scraping Memox page 9: https://memox.co.id/tag/kabupaten-malang/page/9/
Scraping Memox page 10: https://memox.co.id/tag/kabupaten-malang/page/10/
Scraping Memox page 11: https://memox.co.id/tag/kabupaten-malang/page/11/
Scraping Memox page 12: https://memox.co.id/tag/kabupaten-malang/page/12/
page=001 | date=2026-06-26T13:15:52+07:00 | title=Usai Diresmikan Kapolres Malang Ratusan Jamaah Sholat Jumat Datangi Masjid Budi Trianti

,month,count
0,2026-02,1
1,2026-03,44
2,2026-04,24
3,2026-05,30
4,2026-06,31



rows per page:


,page_num,rows,newest,oldest
0,1,10,2026-06-26 13:15:52,2026-06-19 10:14:25
1,2,13,2026-06-18 14:21:21,2026-06-05 16:26:49
2,3,13,2026-06-05 10:00:20,2026-05-26 20:36:41
3,4,13,2026-05-26 20:27:45,2026-05-13 15:40:17
4,5,13,2026-05-13 15:36:58,2026-04-30 12:10:56
5,6,13,2026-04-29 00:32:31,2026-04-13 09:04:46
6,7,13,2026-04-12 21:09:00,2026-03-30 17:09:21
7,8,13,2026-03-28 09:50:30,2026-03-18 18:58:10
8,9,13,2026-03-18 18:54:08,2026-03-12 18:58:19
9,10,13,2026-03-12 18:41:21,2026-03-05 14:26:18


## Save debug list CSV

In [6]:
df_to_save = live_df if 'live_df' in globals() and len(live_df) else local_df
OUTPUT_PATH.parent.mkdir(exist_ok=True)
df_to_save.to_csv(OUTPUT_PATH, index=False)
print('saved:', OUTPUT_PATH)


saved: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/csv/memox_list_debug.csv


## Optional article sample check

In [7]:
sample_rows = (live_df if 'live_df' in globals() and len(live_df) else local_df).head(5)
article_rows = []
for index, row in sample_rows.iterrows():
    try:
        article = scraper.extract_article(row)
        article_rows.append(article)
        print(f"[{len(article_rows)}] content_len={len(article.get('content') or '')} | {short_title(article.get('title'))}")
    except Exception as error:
        print(f"failed sample article {index + 1}: {row.get('url')} | {error}")
article_df = pd.DataFrame(article_rows)
display(article_df[['title', 'published_date', 'content', 'url']].head() if len(article_df) else article_df)


[1] content_len=2158 | Usai Diresmikan Kapolres Malang Ratusan Jamaah Sholat Jumat Datangi Masjid Budi Trianti...
[2] content_len=3752 | Sambut Hari Bhayangkara ke-80 Polres Malang Bangun 7 Sumur Bor di 3 Kecamatan
[3] content_len=2538 | Giat Donor Darah Hari Bhayangkara ke-80 di Polres Malang Diikuti TNI dan Masyarakat Umum
[4] content_len=4218 | Mengaku Ajudan Gubenur Jatim, 2 Tersangka Kasus Penipuan Program UMKM Diamankan Polres ...
[5] content_len=2411 | Jelang Hari Bhayangkara ke-80, Kapolres Malang Pimpin Giat Ziarah dan Tabur Bunga di TMP


,title,published_date,content,url
0,Usai Diresmikan Kapolres Malang Ratusan Jamaah Sholat Jumat Datangi Masjid Budi Trianti Polsek Lawang,2026-06-26,MEMOX.CO.ID\n– Usai diresmikan Kapolres Malang AKBP M Taat Risdi Jum’at (26/06/26) ratusan jamaah sholat Jum’at mendatangi masjid Budi Trianti Polsek Lawang...,https://memox.co.id/usai-diresmikan-kapolres-malang-ratusan-jamaah-sholat-jumat-datangi-masjid-budi-trianti-polsek-lawang/
1,Sambut Hari Bhayangkara ke-80 Polres Malang Bangun 7 Sumur Bor di 3 Kecamatan,2026-06-25,"MEMOX.CO.ID\n– Polres Malang meresmikan tujuh titik bantuan sosial Sumur Bor Polri Presisi yang tersebar di wilayah Kecamatan Pagak, Donomulyo, dan Sumberma...",https://memox.co.id/sambut-hari-bhayangkara-ke-80-polres-malang-bangun-7-sumur-bor-di-3-kecamatan/
2,Giat Donor Darah Hari Bhayangkara ke-80 di Polres Malang Diikuti TNI dan Masyarakat Umum,2026-06-24,"MEMOX.CO.ID\n– Polres Malang menggelar kegiatan donor darah dalam rangka menyambut Hari Bhayangkara ke-80 di GOR Indoor Polres Malang, Rabu (24/6/2026).\nKe...",https://memox.co.id/giat-donor-darah-hari-bhayangkara-ke-80-di-polres-malang-diikuti-tni-dan-masyarakat-umum/
3,"Mengaku Ajudan Gubenur Jatim, 2 Tersangka Kasus Penipuan Program UMKM Diamankan Polres Malang",2026-06-24,MEMOX.CO.ID\n– Satreskrim Polres Malang membongkar dugaan tindak pidana penipuan berkedok program pengembangan UMKM yang juga mengaku ajudan Gubernur Jatim ...,https://memox.co.id/mengaku-ajudan-gubenur-jatim-2-tersangka-kasus-penipuan-program-umkm-diamankan-polres-malang/
4,"Jelang Hari Bhayangkara ke-80, Kapolres Malang Pimpin Giat Ziarah dan Tabur Bunga di TMP",2026-06-23,"MEMOX.CO.ID\n– Polres Malang menggelar ziarah rombongan di Taman Makam Pahlawan (TMP) Prayuda Nirwana, Kecamatan Kepanjen, Kabupaten Malang, dalam rangka me...",https://memox.co.id/jelang-hari-bhayangkara-ke-80-kapolres-malang-pimpin-giat-ziarah-dan-tabur-bunga-di-tmp/
